In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 78 — Context-Augmented Analytics Bot (LlamaIndex)

## Goal
Query **structured data** (tables/SQL) and **unstructured
documents** together through a unified interface.

## What You'll Learn
| Concept | Detail |
|---------|--------|
| **Structured data** | Query SQLite with natural language |
| **Unstructured docs** | RAG over text documents |
| **Router** | LLM decides which data source to query |
| **Combined answers** | Merge insights from both sources |

## Stack
- `llama-index-core` 0.14+
- `llama-index-llms-ollama`
- `llama-index-embeddings-huggingface`
- SQLite for structured data

In [ ]:
import os, warnings, sqlite3
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

Settings.llm = Ollama(model="qwen3.5:9b", request_timeout=120.0, temperature=0)
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
Settings.chunk_size = 256

print("All imports OK")

## Step 1 — Create Structured Data (SQLite)

In [ ]:
# Create analytics database
DB_PATH = "analytics_bot.db"
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

cur.executescript("""
DROP TABLE IF EXISTS sales;
DROP TABLE IF EXISTS products;

CREATE TABLE products (
    id INTEGER PRIMARY KEY,
    name TEXT, category TEXT, price REAL
);

CREATE TABLE sales (
    id INTEGER PRIMARY KEY,
    product_id INTEGER, quarter TEXT, units INTEGER, revenue REAL,
    FOREIGN KEY (product_id) REFERENCES products(id)
);

INSERT INTO products VALUES (1, 'Widget Pro', 'Hardware', 299.99);
INSERT INTO products VALUES (2, 'Widget Lite', 'Hardware', 149.99);
INSERT INTO products VALUES (3, 'CloudSync', 'Software', 49.99);
INSERT INTO products VALUES (4, 'DataVault', 'Software', 99.99);
INSERT INTO products VALUES (5, 'SecureNet', 'Security', 199.99);

INSERT INTO sales VALUES (1, 1, 'Q1-2024', 150, 44998.50);
INSERT INTO sales VALUES (2, 1, 'Q2-2024', 180, 53998.20);
INSERT INTO sales VALUES (3, 1, 'Q3-2024', 200, 59998.00);
INSERT INTO sales VALUES (4, 2, 'Q1-2024', 300, 44997.00);
INSERT INTO sales VALUES (5, 2, 'Q2-2024', 280, 41997.20);
INSERT INTO sales VALUES (6, 2, 'Q3-2024', 350, 52496.50);
INSERT INTO sales VALUES (7, 3, 'Q1-2024', 500, 24995.00);
INSERT INTO sales VALUES (8, 3, 'Q2-2024', 620, 30993.80);
INSERT INTO sales VALUES (9, 3, 'Q3-2024', 700, 34993.00);
INSERT INTO sales VALUES (10, 4, 'Q1-2024', 200, 19998.00);
INSERT INTO sales VALUES (11, 4, 'Q2-2024', 250, 24997.50);
INSERT INTO sales VALUES (12, 4, 'Q3-2024', 230, 22997.70);
INSERT INTO sales VALUES (13, 5, 'Q1-2024', 100, 19999.00);
INSERT INTO sales VALUES (14, 5, 'Q2-2024', 120, 23998.80);
INSERT INTO sales VALUES (15, 5, 'Q3-2024', 90, 17999.10);
""")
conn.commit()

# Verify
cur.execute("SELECT p.name, SUM(s.revenue) FROM products p JOIN sales s ON p.id=s.product_id GROUP BY p.name")
print("Products & Total Revenue:")
for name, rev in cur.fetchall():
    print(f"  {name}: ${rev:,.2f}")

## Step 2 — Create Unstructured Documents (Context)

In [ ]:
# Business context documents
context_docs = [
    Document(text="""Q3 2024 Market Analysis
The hardware market saw strong growth driven by enterprise upgrades.
Widget Pro gained traction in mid-market companies due to its reliability.
Widget Lite performed well in education and small business segments.
Competition from Asian manufacturers is increasing price pressure.""",
             metadata={"type": "report", "quarter": "Q3-2024"}),
    
    Document(text="""Software Product Strategy 2024
CloudSync is our fastest-growing product, targeting SaaS companies needing
real-time data synchronization. The freemium-to-paid conversion rate is 12%.
DataVault appeals to compliance-heavy industries (finance, healthcare).
Planned: AI-powered features for CloudSync in Q4, enterprise tier for DataVault.""",
             metadata={"type": "strategy", "quarter": "2024"}),
    
    Document(text="""SecureNet Product Overview
SecureNet is our network security appliance. Target: SMBs without dedicated
security teams. Q3 sales declined 10% due to a product recall in August
(firmware vulnerability). The fix was deployed in September. Customer
satisfaction scores dropped from 4.2 to 3.8 during the recall period.""",
             metadata={"type": "product", "quarter": "Q3-2024"}),
    
    Document(text="""Sales Team Performance Notes
The enterprise sales team closed 3 major deals in Q3, all involving Widget Pro
bundles with CloudSync. Average enterprise deal size: $45,000. The SMB team
struggled with longer sales cycles for SecureNet after the recall.
Recommendation: increase marketing spend on CloudSync and offer SecureNet
discounts to rebuild trust.""",
             metadata={"type": "internal", "quarter": "Q3-2024"}),
]

# Build vector index
doc_index = VectorStoreIndex.from_documents(context_docs)
doc_engine = doc_index.as_query_engine(similarity_top_k=2)

print(f"Indexed {len(context_docs)} context documents")

## Step 3 — Build the Analytics Bot

The bot decides whether to query **SQL** (numbers),
**documents** (context), or **both**.

In [ ]:
# Safe SQL executor (read-only)
BLOCKED = {"INSERT", "UPDATE", "DELETE", "DROP", "ALTER", "CREATE", "TRUNCATE"}

def run_sql(query: str) -> str:
    upper = query.upper().strip()
    if any(cmd in upper for cmd in BLOCKED):
        return "ERROR: Only SELECT queries allowed."
    try:
        c = sqlite3.connect(DB_PATH)
        cur = c.cursor()
        cur.execute(query)
        cols = [d[0] for d in cur.description] if cur.description else []
        rows = cur.fetchall()
        c.close()
        if not rows:
            return "No results."
        result = " | ".join(cols) + "\n"
        for row in rows:
            result += " | ".join(str(v) for v in row) + "\n"
        return result
    except Exception as e:
        return f"SQL Error: {e}"

def analytics_bot(question: str):
    """Route question to SQL, docs, or both."""
    print(f"\n{'='*70}")
    print(f"Q: {question}")
    print(f"{'='*70}")
    
    llm = Settings.llm
    
    # Step 1: Route the question
    route_prompt = f"""Classify this question into one category:
- 'sql': needs numerical data from database (revenue, units, counts)
- 'docs': needs qualitative context (strategy, analysis, reasons)
- 'both': needs numbers AND context

Question: {question}

Reply with exactly one word: sql, docs, or both."""
    
    route = str(llm.complete(route_prompt)).strip().lower()
    # Extract just the category word
    for cat in ["both", "sql", "docs"]:
        if cat in route:
            route = cat
            break
    else:
        route = "both"
    print(f"  📡 Route: {route}")
    
    sql_data = ""
    doc_data = ""
    
    # Step 2: Get SQL data if needed
    if route in ("sql", "both"):
        sql_prompt = f"""Given these tables:
- products(id, name, category, price)
- sales(id, product_id, quarter, units, revenue)

Write a SQL query to answer: {question}
Reply with ONLY the SQL query, no explanation."""
        sql_query = str(llm.complete(sql_prompt)).strip()
        # Clean up: remove markdown code blocks
        sql_query = sql_query.replace("```sql", "").replace("```", "").strip()
        # Take first line if multiple
        sql_query = sql_query.split("\n")[0].strip()
        print(f"  🗄️ SQL: {sql_query}")
        sql_data = run_sql(sql_query)
        print(f"  📊 Result: {sql_data[:200]}")
    
    # Step 3: Get doc context if needed
    if route in ("docs", "both"):
        doc_response = doc_engine.query(question)
        doc_data = str(doc_response)
        print(f"  📄 Doc context: {doc_data[:200]}...")
    
    # Step 4: Synthesize answer
    synth_prompt = f"""Answer the question using the provided data.

{'SQL Data:\n' + sql_data if sql_data else ''}
{'Document Context:\n' + doc_data if doc_data else ''}

Question: {question}

Provide a clear, data-backed answer."""
    
    answer = str(llm.complete(synth_prompt))
    print(f"\n💡 Answer:\n{answer[:600]}")
    return answer

print("Analytics bot ready!")

In [ ]:
# Test 1: SQL question
analytics_bot("What is the total revenue by product category?")

In [ ]:
# Test 2: Document question
analytics_bot("Why did SecureNet sales decline in Q3?")

In [ ]:
# Test 3: Combined question
analytics_bot("Which product has the highest growth and what is the strategy for it?")

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **Query routing** | LLM classifies question as sql/docs/both |
| **Text-to-SQL** | LLM generates SQL from natural language |
| **RAG** | Vector search over context documents |
| **Answer synthesis** | Combine data from both sources |

## Architecture
```
User Question
    │
    ├─ Router (LLM classifies: sql / docs / both)
    │
    ├─ SQL Path:  LLM → SQL query → SQLite → results
    │
    ├─ Doc Path:  Query → Vector index → relevant docs
    │
    └─ Synthesizer (LLM combines SQL data + doc context → answer)
```

## Extending This Notebook
- Use LlamaIndex `SQLTableRetrieverQueryEngine` for auto SQL
- Add charts with matplotlib for numeric results
- Cache frequent queries to avoid redundant LLM calls
- Add time-series analysis tools